# Boltz-2 구조 예측 확장 — ESM2 미니 프로젝트 Standard 범위

Phase 1(ESM2 벤치마크 검증), Phase 2(파이타아제/알파-아밀라아제 saturation mutagenesis)에 이어서,
**야생형(WT) vs ESM2가 뽑은 상위 후보 변이체의 구조를 Boltz-2로 예측하고 비교**합니다.

Boltz-2는 AlphaFold3와 같은 계열의 오픈소스 구조 예측 모델로, 결합 친화도(binding affinity)까지 예측할 수 있다는 점이 특징입니다 (MIT 라이선스, [GitHub](https://github.com/jwohlwend/boltz)).

**주의**: 이 노트북은 Colab에서 GPU(가능하면 A100, 최소 T4)로 실행하는 것을 전제로 작성했습니다. Boltz-2는 ESM2보다 훨씬 무겁고, 구조 예측 1개당 MSA 조회 + 모델 추론까지 몇 분씩 걸릴 수 있습니다. 이 부분은 이 세션에서 직접 실행해보지 못했으니(GPU/네트워크 제약), 먼저 아래 '스모크 테스트'로 환경이 정상 동작하는지 확인한 뒤 본 실행으로 넘어가세요.


## 0. 설치

In [ ]:
!pip install -q "boltz[cuda]" -U
!boltz --help


## 1. 스모크 테스트 (작은 서열로 먼저 확인)

전체 파이프라인을 큰 서열로 돌리기 전에, 짧은 임의 서열로 `boltz predict`가 정상 동작하는지 먼저 확인합니다.
`--use_msa_server`를 쓰면 MSA를 직접 준비할 필요 없이 Boltz가 자동으로 조회합니다 (ColabFold의 MMseqs2 서버 이용, 네트워크 필요).


In [ ]:
import os
os.makedirs("boltz_smoke_test", exist_ok=True)

smoke_yaml = '''version: 1
sequences:
  - protein:
      id: A
      sequence: MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWELVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL
'''
with open("boltz_smoke_test/test.yaml", "w") as f:
    f.write(smoke_yaml)

!boltz predict boltz_smoke_test/test.yaml --use_msa_server --out_dir boltz_smoke_test/out


정상적으로 돌아갔다면 `boltz_smoke_test/out` 아래에 구조 파일(.cif)과 신뢰도(confidence) 정보가 담긴 json이 생겼을 거예요. 에러가 나면 대부분 (1) GPU 미할당 (2) 네트워크 문제(MSA 서버 접속 실패) (3) boltz 버전 이슈 중 하나입니다. `boltz predict --help`로 옵션을 확인하고, 필요하면 이 셀 에러 메시지를 저에게 붙여넣어주세요.


## 2. WT vs 상위 후보 변이체 서열 준비

Phase 2에서 저장한 saturation mutagenesis 결과(`phase2_*_saturation_mutagenesis.csv`)에서 상위 후보를 불러와서, 그 변이를 WT 서열에 실제로 적용한 변이체 서열을 만듭니다.


In [ ]:
import pandas as pd

def apply_mutation(sequence: str, mutant: str) -> str:
    wt_aa, pos, mut_aa = mutant[0], int(mutant[1:-1]), mutant[-1]
    assert sequence[pos - 1] == wt_aa, f"{mutant}: 서열의 {pos}번 잔기가 {sequence[pos-1]}이라 {wt_aa}와 불일치"
    return sequence[:pos - 1] + mut_aa + sequence[pos:]

TOP_N = 3  # 효소당 비교할 상위 후보 개수

targets = {
    "phytase_PHYA_ASPNG_P34752": ENZYMES["phytase_PHYA_ASPNG_P34752"]["sequence"],
    "alpha_amylase_AMY_BACLI_P06278": ENZYMES["alpha_amylase_AMY_BACLI_P06278"]["sequence"],
}

variant_jobs = []  # (enzyme_name, job_id, sequence)
for name, wt_seq in targets.items():
    variant_jobs.append((name, "WT", wt_seq))
    sat_df = pd.read_csv(f"phase2_{name}_saturation_mutagenesis.csv")
    top = sat_df.sort_values("esm2_llr", ascending=False).head(TOP_N)
    for _, row in top.iterrows():
        mut_seq = apply_mutation(wt_seq, row["mutant"])
        variant_jobs.append((name, row["mutant"], mut_seq))

for name, job_id, seq in variant_jobs:
    print(f"{name:35s} {job_id:8s} len={len(seq)}")


## 3. Boltz-2 입력(YAML) 생성 및 구조 예측 실행

각 서열(WT + 상위 변이체)마다 YAML 파일을 만들고, `boltz predict`로 구조를 예측합니다. `--use_msa_server`로 MSA를 자동 생성하므로 로컬 MSA 파일 없이 서열만 있으면 됩니다.


In [ ]:
import os

BOLTZ_ROOT = "boltz_runs"
os.makedirs(BOLTZ_ROOT, exist_ok=True)

for name, job_id, seq in variant_jobs:
    job_dir = f"{BOLTZ_ROOT}/{name}/{job_id}"
    os.makedirs(job_dir, exist_ok=True)
    yaml_content = f'''version: 1
sequences:
  - protein:
      id: A
      sequence: {seq}
'''
    with open(f"{job_dir}/input.yaml", "w") as f:
        f.write(yaml_content)

print(f"총 {len(variant_jobs)}개 YAML 생성 완료")


In [ ]:
# 구조 예측 실행 (효소당 WT+3개 변이 = 4개 job, 시간이 걸립니다)
for name, job_id, seq in variant_jobs:
    job_dir = f"{BOLTZ_ROOT}/{name}/{job_id}"
    print(f"=== {name} / {job_id} ===")
    !boltz predict {job_dir}/input.yaml --use_msa_server --out_dir {job_dir}/out


## 4. 결과 비교: WT vs 변이체 구조 신뢰도

Boltz는 각 예측에 대해 신뢰도 지표(예: pLDDT, confidence score)를 json으로 함께 출력합니다. 파일 경로/키 이름은 boltz 버전에 따라 조금씩 다를 수 있으니, 아래 셀에서 실제 출력 폴더 구조를 먼저 확인하세요.


In [ ]:
import glob, json

# 실제 출력 폴더 구조 확인 (버전에 따라 파일명이 다를 수 있음)
for name, job_id, seq in variant_jobs[:1]:
    job_dir = f"{BOLTZ_ROOT}/{name}/{job_id}/out"
    for path in glob.glob(f"{job_dir}/**/*", recursive=True):
        print(path)


In [ ]:
# 위에서 확인한 실제 파일 경로 패턴에 맞춰 신뢰도 점수를 모으세요.
# 예시 (boltz 최신 버전 기준, 실제 파일명은 위 셀 출력을 보고 맞춰서 수정):
confidence_records = []
for name, job_id, seq in variant_jobs:
    job_dir = f"{BOLTZ_ROOT}/{name}/{job_id}/out"
    conf_files = glob.glob(f"{job_dir}/**/confidence*.json", recursive=True)
    if not conf_files:
        print(f"[경고] {name}/{job_id}: confidence json을 못 찾음, 위 셀 출력을 보고 경로를 확인하세요")
        continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    confidence_records.append({"enzyme": name, "job_id": job_id, **conf})

conf_df = pd.DataFrame(confidence_records)
conf_df


## 5. (선택/스트레치) 결합 친화도 예측

자연 기질과의 결합 친화도까지 비교하려면, 리간드를 SMILES로 추가하면 됩니다. **정확한 SMILES는 직접 [PubChem](https://pubchem.ncbi.nlm.nih.gov)에서 검색해서 넣으세요** (제가 이 세션에서는 PubChem 접근이 막혀서 직접 검증하지 못했습니다 — 틀린 SMILES를 그냥 쓰는 것보다 안전합니다).

- 파이타아제 기질: phytic acid (myo-inositol hexakisphosphate) 검색
- 아밀라아제 기질/유사체: maltose, maltotriose, 또는 acarbose(억제제) 검색

YAML에 리간드와 `affinity` 속성을 추가하는 형식:
```yaml
version: 1
sequences:
  - protein:
      id: A
      sequence: <효소 서열>
  - ligand:
      id: B
      smiles: '<PubChem에서 가져온 SMILES>'
properties:
  - affinity:
      binder: B
```
리간드는 heavy atom 128개 이하여야 합니다 (phytic acid, 말토올리고당류는 문제없음).


In [ ]:
# SMILES를 확인한 뒤 아래를 채우고 실행하세요.
LIGAND_SMILES = {
    "phytase_PHYA_ASPNG_P34752": None,      # PubChem에서 phytic acid SMILES 확인 후 채우기
    "alpha_amylase_AMY_BACLI_P06278": None,  # PubChem에서 maltose/acarbose SMILES 확인 후 채우기
}

for name, job_id, seq in variant_jobs:
    smiles = LIGAND_SMILES.get(name)
    if not smiles:
        continue
    job_dir = f"{BOLTZ_ROOT}_affinity/{name}/{job_id}"
    os.makedirs(job_dir, exist_ok=True)
    yaml_content = f'''version: 1
sequences:
  - protein:
      id: A
      sequence: {seq}
  - ligand:
      id: B
      smiles: '{smiles}'
properties:
  - affinity:
      binder: B
'''
    with open(f"{job_dir}/input.yaml", "w") as f:
        f.write(yaml_content)
    print(f"=== affinity: {name} / {job_id} ===")
    !boltz predict {job_dir}/input.yaml --use_msa_server --out_dir {job_dir}/out


## 6. 정리

- WT 대비 상위 후보 변이체들의 구조적 신뢰도(pLDDT 등)가 유지되는지 확인 — 크게 떨어진다면 그 변이가 구조를 불안정하게 만들 가능성 신호
- (선택) 결합 친화도까지 봤다면, ESM2가 뽑은 "진화적으로 자연스러운" 변이가 실제로 기질과의 결합을 개선/유지하는지 추가 근거로 활용
- 이 단계까지 오면 자소서에 "서열(ESM2) + 구조(Boltz-2) 두 layer의 protein AI를 결합한 in-silico 효소 설계 파이프라인"이라고 쓸 수 있음 — 공고의 "단백질 서열 공간 탐색 + 구조/물성 예측"이라는 두 문장을 동시에 커버
